# Demand Forecasting Dashboard
**Marketing Analytics Module 1** — Naïve baseline + 3 ML models with category pooling

Models: Naïve (Moving Avg) · Linear Regression · Random Forest · MLP Neural Network  
Features: 5 economically motivated variables (price, promo, trend, quarter, baseline demand)  
Training: Category-pooled data for ML models  
Evaluation: temporal train/test split (no leakage)

In [1]:
# Load all the libraries needed for data handling, charting, and forecasting models
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from IPython.display import display, HTML
import ipywidgets as widgets
import warnings
warnings.filterwarnings("ignore")

pio.templates.default = "plotly_white"

# Assign a consistent color to each model so charts are easy to read
COLORS = {
    "Naïve (Moving Avg)": "#64748b",       # grey — baseline benchmark
    "Linear Regression": "#7C3AED",         # purple — matches dashboard
    "Random Forest": "#E11D48",             # red — matches dashboard
    "Neural Network (MLP)": "#EA580C",      # orange — matches dashboard
    "actual": "#16a34a",                     # green — matches dashboard
    "ci": "rgba(99,102,241,0.12)",
}

# Build the styled KPI summary cards shown at the top of each SKU analysis

def metric_card(label, value, sub=""):
    return f'''<div style="flex:1;min-width:160px;background:#f8fafc;border:1px solid #e2e8f0;
    border-radius:12px;padding:1.1rem 1.4rem;">
    <div style="font-size:0.72rem;color:#64748b;text-transform:uppercase;letter-spacing:0.05em;font-weight:500;">{label}</div>
    <div style="font-size:1.45rem;font-weight:700;color:#0f172a;margin-top:0.2rem;font-family:monospace;">{value}</div>
    <div style="font-size:0.78rem;color:#64748b;margin-top:0.15rem;">{sub}</div></div>'''

def metric_row(cards):
    inner = "".join(cards)
    return HTML(f'<div style="display:flex;gap:1rem;margin:1rem 0;flex-wrap:wrap;">{inner}</div>')

print("Imports loaded")


Imports loaded


In [2]:
# Read the raw sales data and the pre-processed (feature-engineered) version
raw = pd.read_csv("../data/data_raw.csv")
proc = pd.read_csv("../data/data_processed.csv")
raw["week"] = pd.to_datetime(raw["week"])
proc["week"] = pd.to_datetime(proc["week"])
raw["feat_main_page"] = raw["feat_main_page"].astype(str).str.lower().eq("true").astype(int)

# Summarise each SKU: its category, color, average price, average weekly sales, and promo rate
sku_meta = raw.groupby("sku").agg(
    functionality=("functionality", "first"),
    color=("color", "first"),
    avg_price=("price", "mean"),
    avg_sales=("weekly_sales", "mean"),
    feat_rate=("feat_main_page", "mean"),
).reset_index()
sku_meta["avg_price"] = sku_meta["avg_price"].round(2)
sku_meta["avg_sales"] = sku_meta["avg_sales"].round(1)
sku_meta["feat_rate"] = (sku_meta["feat_rate"] * 100).round(1)

print(f"Raw:       {raw.shape[0]:,} rows × {raw.shape[1]} cols — {raw.sku.nunique()} SKUs × {raw.week.nunique()} weeks")
print(f"Processed: {proc.shape[0]:,} rows × {proc.shape[1]} cols")
sku_meta[["sku","functionality","color","avg_price","avg_sales","feat_rate"]].head(10)


Raw:       4,400 rows × 8 cols — 44 SKUs × 100 weeks
Processed: 4,312 rows × 48 cols


,sku,functionality,color,avg_price,avg_sales,feat_rate
0,1,06.Mobile phone accessories,black,24.01,22.2,21.0
1,2,07.Headphones,blue,64.51,8.5,8.0
2,3,07.Headphones,purple,103.58,10.6,14.0
3,4,06.Mobile phone accessories,red,19.46,8.9,50.0
4,5,06.Mobile phone accessories,red,10.94,9.4,72.0
5,6,04.Selfie sticks,blue,36.20,16.6,10.0
6,7,04.Selfie sticks,blue,11.27,85.2,6.0
7,8,03.Bluetooth speakers,black,113.01,31.2,62.0
8,9,11.Fitness trackers,black,161.80,73.6,79.0
9,10,10.VR headset,white,188.74,14.6,75.0


In [3]:
# ── Feature builder: 5 economically motivated features ──
# price, promo, trend, quarter, price_change — 5 features on ~90 rows = 18:1 ratio.
#
# price_change = price - price_lag1 captures the reference-price effect
# ("did price just go up or down?") without the multicollinearity of raw lags.
# Raw price levels (price, price-1, price-2) are ~95% correlated (VIF > 50),
# but current price and the *change* from last week are nearly uncorrelated.

DEMAND_FEATURES = ['price', 'feat_main_page', 'trend', 'quarter', 'price_change']

def build_demand_features(proc, sku_id):
    """Build minimal, economically motivated feature set for one SKU."""
    df = proc[proc['sku'] == sku_id].sort_values('week').copy()
    X = pd.DataFrame(index=df.index)
    X['price'] = df['price'].values
    X['feat_main_page'] = df['feat_main_page'].values
    X['trend'] = df['trend'].values
    X['quarter'] = pd.to_datetime(df['week']).dt.quarter.values
    if 'price-1' in df.columns:
        lag1 = df['price-1'].values.copy()
        lag1 = np.where(lag1 > 0, lag1, df['price'].values)
        X['price_change'] = df['price'].values - lag1
    else:
        X['price_change'] = 0.0
    return X.values, df['weekly_sales'].values, df['week'].values, list(X.columns)

print(f'Features: {DEMAND_FEATURES}')

Features: ['price', 'feat_main_page', 'trend', 'quarter', 'price_change']


## Hyperparameter tuning

Tuning runs on **category-pooled data with 5 features** — the same feature space the models will use. `TimeSeriesSplit` (3 folds) respects temporal ordering.

In [4]:
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV

sku_sales = proc.groupby('sku')['weekly_sales'].mean().sort_values()
sample_skus = [sku_sales.index[i] for i in np.linspace(0, len(sku_sales)-1, 6, dtype=int)]

X_pool, y_pool = [], []
for s in sample_skus:
    X_s, y_s, _, _ = build_demand_features(proc, s)
    X_pool.append(X_s[:-8])
    y_pool.append(y_s[:-8])
X_pool = np.vstack(X_pool)
y_pool = np.concatenate(y_pool)

tscv = TimeSeriesSplit(n_splits=3)

rf_search = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    {'n_estimators': [100, 200], 'max_depth': [4, 6, 8], 'min_samples_leaf': [2, 4, 8]},
    cv=tscv, scoring='neg_mean_absolute_error', n_jobs=-1)
rf_search.fit(X_pool, y_pool)
TUNED_RF = rf_search.best_params_
print(f'RF best:  {TUNED_RF}  (CV MAE: {-rf_search.best_score_:.2f})')

scaler_tune = StandardScaler()
X_pool_s = scaler_tune.fit_transform(X_pool)
mlp_search = GridSearchCV(
    MLPRegressor(max_iter=500, early_stopping=True, validation_fraction=0.15, random_state=42),
    {'hidden_layer_sizes': [(32,), (64, 32)], 'learning_rate_init': [0.001, 0.005, 0.01]},
    cv=tscv, scoring='neg_mean_absolute_error', n_jobs=-1)
mlp_search.fit(X_pool_s, y_pool)
TUNED_MLP = mlp_search.best_params_
print(f'MLP best: {TUNED_MLP}  (CV MAE: {-mlp_search.best_score_:.2f})')
print(f'Tuned on {X_pool.shape[0]} rows × {X_pool.shape[1]} features')

RF best:  {'max_depth': 4, 'min_samples_leaf': 4, 'n_estimators': 100}  (CV MAE: 217.73)
MLP best: {'hidden_layer_sizes': (64, 32), 'learning_rate_init': 0.001}  (CV MAE: 226.93)
Tuned on 540 rows × 5 features


In [5]:
# ── Core modelling: naïve baseline + 3 ML models (per-SKU training) ──

def train_evaluate(X_sku, y_sku, weeks, feature_cols, test_size):
    """Train naïve + 3 ML models on this SKU's data, evaluate on held-out test."""
    X_train = X_sku[:-test_size]
    X_test = X_sku[-test_size:]
    y_train = y_sku[:-test_size]
    y_test = y_sku[-test_size:]
    w_train, w_test = weeks[:-test_size], weeks[-test_size:]

    # Naïve baseline: average of last 8 training weeks
    lookback = min(8, len(y_train))
    naive_level = float(np.mean(y_train[-lookback:]))
    naive_std = float(np.std(y_train[-lookback:])) if lookback > 1 else float(np.std(y_train))

    results = {}
    results['Naïve (Moving Avg)'] = {
        'model': None, 'scaler': None,
        'y_pred_test': np.full(test_size, naive_level),
        'resid_std': naive_std,
        'mae': mean_absolute_error(y_test, np.full(test_size, naive_level)),
        'rmse': np.sqrt(mean_squared_error(y_test, np.full(test_size, naive_level))),
        'r2': r2_score(y_test, np.full(test_size, naive_level)),
        'mape': np.mean(np.abs((y_test - naive_level) / np.maximum(y_test, 1))) * 100,
    }

    # ML models on this SKU's training data
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    ml_models = {
        'Linear Regression': LinearRegression(),
        'Random Forest': RandomForestRegressor(**TUNED_RF, random_state=42, n_jobs=-1),
        'Neural Network (MLP)': MLPRegressor(
            hidden_layer_sizes=TUNED_MLP['hidden_layer_sizes'],
            learning_rate_init=TUNED_MLP['learning_rate_init'],
            max_iter=500, early_stopping=True, validation_fraction=0.15, random_state=42),
    }

    for name, model in ml_models.items():
        use_scaled = 'Neural' in name
        Xtr = X_train_s if use_scaled else X_train
        Xte = X_test_s if use_scaled else X_test
        model.fit(Xtr, y_train)
        y_pred_test = np.maximum(model.predict(Xte), 0)
        resid_std = np.std(y_train - np.maximum(model.predict(Xtr), 0))
        results[name] = {
            'model': model, 'scaler': scaler if use_scaled else None,
            'y_pred_test': y_pred_test,
            'resid_std': resid_std,
            'mae': mean_absolute_error(y_test, y_pred_test),
            'rmse': np.sqrt(mean_squared_error(y_test, y_pred_test)),
            'r2': r2_score(y_test, y_pred_test),
            'mape': np.mean(np.abs((y_test - y_pred_test) / np.maximum(y_test, 1))) * 100,
        }

    return results, y_train, y_test, w_train, w_test


def forecast_future(results, X_sku, feature_cols, n_periods=4):
    """Generate future forecasts from each model."""
    last_row = X_sku[-1].copy().reshape(1, -1)
    forecasts = {}
    for name, res in results.items():
        preds, ci_lower, ci_upper = [], [], []
        if res['model'] is None:  # naïve
            level = float(res['y_pred_test'][0])
            for step in range(1, n_periods + 1):
                ci_w = res['resid_std'] * 1.96 * np.sqrt(step)
                preds.append(level)
                ci_lower.append(max(level - ci_w, 0))
                ci_upper.append(level + ci_w)
        else:
            for step in range(1, n_periods + 1):
                row = last_row.copy()
                if 'trend' in feature_cols:
                    row[0, feature_cols.index('trend')] = X_sku[-1, feature_cols.index('trend')] + step / len(X_sku)
                Xf = res['scaler'].transform(row) if res['scaler'] else row
                pred = max(res['model'].predict(Xf)[0], 0)
                ci_w = res['resid_std'] * 1.96 * np.sqrt(step)
                preds.append(pred)
                ci_lower.append(max(pred - ci_w, 0))
                ci_upper.append(pred + ci_w)
        forecasts[name] = {
            'predictions': np.array(preds),
            'ci_lower': np.array(ci_lower),
            'ci_upper': np.array(ci_upper),
        }
    return forecasts

print('Model functions defined')

Model functions defined


In [6]:
# ── Plotting functions ──

# Draw the main forecast chart: historical sales, test-period actuals, model predictions, and future forecast
def plot_forecast(w_train, y_train, w_test, y_test, results, forecasts, best_model, sku_id, n_periods):
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=w_train, y=y_train, mode="lines", name="Historical",
        line=dict(color="#00008B", width=1.8),
        hovertemplate="Week: %{x|%Y-%m-%d}<br>Sales: %{y:.0f}<extra></extra>",
    ))
    fig.add_trace(go.Scatter(
        x=w_test, y=y_test, mode="lines+markers", name="Actual (test)",
        line=dict(color=COLORS["actual"], width=2.5), marker=dict(size=6, color=COLORS["actual"]),
        hovertemplate="Week: %{x|%Y-%m-%d}<br>Sales: %{y:.0f}<extra></extra>",
    ))

    last_date = pd.Timestamp(w_test[-1])
    future_weeks = pd.date_range(start=last_date + pd.Timedelta(weeks=1), periods=n_periods, freq="W-MON")

    for name, res in results.items():
        is_best = name == best_model
        opacity = 1.0 if is_best else 0.4
        width = 2.5 if is_best else 1.5
        dash = None if is_best else "dot"

        fig.add_trace(go.Scatter(
            x=w_test, y=res["y_pred_test"], mode="lines",
            name=f"{name} (test)", line=dict(color=COLORS[name], width=width, dash=dash),
            opacity=opacity,
        ))

        fc = forecasts[name]
        fig.add_trace(go.Scatter(
            x=future_weeks, y=fc["predictions"], mode="lines+markers",
            name=f"{name} (forecast)", line=dict(color=COLORS[name], width=width),
            marker=dict(size=7 if is_best else 4, symbol="diamond"), opacity=opacity,
        ))

        if is_best:
            fig.add_trace(go.Scatter(
                x=list(future_weeks) + list(future_weeks[::-1]),
                y=list(fc["ci_upper"]) + list(fc["ci_lower"][::-1]),
                fill="toself", fillcolor=COLORS["ci"],
                line=dict(color="rgba(0,0,0,0)"), name="95% CI", showlegend=True,
            ))

    fig.add_shape(type="line", x0=last_date, x1=last_date, y0=0, y1=1,
                  yref="paper", line=dict(width=1.5, dash="dash", color="#94a3b8"))
    fig.add_annotation(x=last_date, y=1, yref="paper", text="Forecast →",
                       showarrow=False, font=dict(size=11, color="#64748b"),
                       xanchor="left", yanchor="top")

    fig.update_layout(
        title=f"SKU {sku_id} — Demand forecast", height=480,
        hovermode="x unified",
        legend=dict(orientation="h", yanchor="bottom", y=-0.25, xanchor="center", x=0.5, font_size=11),
        margin=dict(l=50, r=30, t=60, b=80),
        xaxis_title="Week", yaxis_title="Weekly sales (units)",
    )
    return fig


# Bar chart comparing the three models on MAE, RMSE, and R² (lower error = better, higher R² = better)
def plot_model_comparison(results):
    """Bar chart comparing all models (including naïve) on MAE, RMSE, R²."""
    names = list(results.keys())
    fig = make_subplots(rows=1, cols=3, subplot_titles=["MAE ↓", "RMSE ↓", "R² ↑"])
    for i, metric in enumerate(["mae", "rmse", "r2"]):
        vals = [results[n][metric] for n in names]
        fig.add_trace(go.Bar(
            x=names, y=vals, marker_color=[COLORS[n] for n in names], showlegend=False,
            text=[f"{v:.2f}" for v in vals], textposition="outside",
            textfont=dict(size=11, family="monospace"),
        ), row=1, col=i + 1)
    fig.update_layout(height=300, margin=dict(l=40, r=20, t=40, b=20))
    return fig


# Show prediction errors per week: positive = model underestimated, negative = overestimated
def plot_residuals(w_test, y_test, results, best_model):
    res = results[best_model]
    residuals = y_test - res["y_pred_test"]
    fig = go.Figure(go.Bar(
        x=w_test, y=residuals,
        marker_color=["#6366f1" if r >= 0 else "#ef4444" for r in residuals],
    ))
    fig.add_hline(y=0, line_color="#94a3b8", line_width=1)
    fig.update_layout(title=f"Residuals — {best_model}", height=280,
                      margin=dict(l=40, r=20, t=40, b=20),
                      xaxis_title="Week", yaxis_title="Actual − Predicted")
    return fig


# Rank which inputs (price, trend, promo, etc.) matter most for the Random Forest's predictions
def plot_feature_importance(results, feature_cols, top_n=12):
    rf = results.get("Random Forest", {}).get("model")
    if rf is None:
        fig = go.Figure()
        fig.update_layout(title="No RF model available", height=350)
        return fig
    imp = rf.feature_importances_
    idx = np.argsort(imp)[-top_n:]
    fig = go.Figure(go.Bar(
        x=imp[idx], y=[feature_cols[i] for i in idx],
        orientation="h", marker_color="#6366f1",
    ))
    fig.update_layout(title="Feature importance (Random Forest)", height=350,
                      margin=dict(l=180, r=20, t=40, b=20), xaxis_title="Importance")
    return fig

print("Plotting functions defined")


Plotting functions defined


## Interactive SKU analysis
**Tip:** Click and drag on the chart to zoom in. Double-click to reset.

Select a SKU and configure the forecast below.

### How the forecast works

For each SKU, the 98 weekly observations are split temporally — the last N weeks (default 8) are held out as a test set, and everything before is used for training. Three models are fitted and compared:

| Model | Preprocessing | Why |
|-------|--------------|-----|
| **Linear Regression** | Raw features | OLS solves a closed-form matrix equation — the solution is scale-invariant, so normalising inputs would change coefficient magnitudes but not predictions or fit. |
| **Random Forest** | Raw features | Tree-based models split on thresholds (e.g. "is price > 5.2?"). They only care about the rank ordering of values, not their absolute scale, so scaling has no effect on the splits or predictions. |
| **MLP Neural Network** | StandardScaler (mean=0, std=1) | Neural networks learn via gradient descent — they update weights by small steps proportional to the gradient. If features have vastly different scales (e.g. price 0–50 vs trend 0–1), gradients are dominated by the large-scale feature, causing slow or unstable convergence. Normalising ensures all features contribute equally to learning. This is standard practice for any gradient-based model (Bishop, 2006). |

The best model is auto-selected by lowest test MAE.

### Forecast projection

To generate future predictions beyond the last observed week, the model needs feature values for weeks that haven't happened yet. Most features (price, promotion flag, month dummies) are held at their last known values — effectively assuming "no change." The **trend variable** is the exception: it is incremented by 1/n_obs per forecast step. This is because trend is normalised to span 0→1 across the 98 observed weeks (each week = 1/98 ≈ 0.0102), so adding the same increment per future step continues the long-run trajectory at its historical rate. This is the simplest credible assumption — it says "demand growth (or decline) continues at the pace we've seen," without assuming acceleration or deceleration.

### Confidence intervals

The bands around each forecast use: **σ_residual × 1.96 × √(step)**, where:

- **σ_residual** = standard deviation of the model's errors on the training set. This measures how much the model typically misses by on data it has already seen — it's our baseline measure of forecast noise.
- **1.96** = the z-score for 95% coverage under a normal distribution. Assuming residuals are approximately Gaussian (reasonable for aggregated weekly sales), 95% of future outcomes should fall within ±1.96 standard deviations.
- **√(step)** = the key insight — *why does uncertainty grow with the forecast horizon?* If one-step-ahead forecast errors have variance σ², and errors across successive steps are roughly independent, then the cumulative uncertainty after h steps has variance proportional to h × σ². Taking the square root gives standard deviation σ × √h. This is an exact result for random walk processes, and a standard approximation for regression-based forecasts in time series (Hyndman & Athanasopoulos, 2021, §5.5). The practical meaning: a 1-week-ahead forecast is more reliable than a 4-week-ahead forecast, and a 4-week-ahead forecast is more reliable than a 12-week-ahead forecast — but the degradation slows down (√4 = 2, √12 ≈ 3.5, not 4 and 12).

**Limitations of this approach:** The √(step) approximation assumes errors are independent across steps, which may not hold if there are autocorrelated demand shocks (e.g. a supply disruption that persists for several weeks). In such cases, uncertainty could grow faster than √(step). A more sophisticated approach would use bootstrapped prediction intervals or explicit time series models (e.g. ARIMA), but for a regression-based dashboard with weekly granularity, the residual-based approximation is a practical and widely accepted choice.

**References:**
- Bishop, C.M. (2006). *Pattern Recognition and Machine Learning*. Springer. §5.1 — Feature normalisation for neural networks.
- Cohen, M. C., Gras, P.E., Pentecoste, A., & Zhang, R. (2022). *Demand Prediction in Retail*. Springer Series in Supply Chain Management 14, 1–155.
- Hyndman, R.J. & Athanasopoulos, G. (2021). *Forecasting: Principles and Practice*, 3rd edition. OTexts. §5.5 — Prediction intervals.

In [7]:
# ── Interactive controls: pick a SKU, set forecast horizon, and click Run ──
sku_list = sorted(proc["sku"].unique())
sku_labels = {s: f"SKU {s} — {sku_meta[sku_meta.sku==s].functionality.values[0]}" for s in sku_list}

try:
    _forecast_output.close()
except NameError:
    pass

_forecast_sku = widgets.Dropdown(options=[(sku_labels[s], s) for s in sku_list], value=1, description="SKU:")
_forecast_horizon = widgets.IntSlider(value=4, min=2, max=12, step=1, description="Weeks:")
_forecast_test = widgets.IntSlider(value=8, min=4, max=16, step=1, description="Test size:")
_forecast_btn = widgets.Button(description="Run forecast", button_style="success", icon="play")
_forecast_output = widgets.Output()

# When the button is clicked: train models, forecast, and display all charts and tables
def _run_forecast(_):
    _forecast_output.clear_output()
    with _forecast_output:
        sku_id = _forecast_sku.value
        n_periods = _forecast_horizon.value
        test_size = _forecast_test.value

        # Build features
        X_sku, y, weeks, fcols = build_demand_features(proc, sku_id)
        if len(y) < test_size + 10:
            print(f"⚠ Insufficient data for SKU {sku_id}")
            return

        results, y_train, y_test, w_train, w_test_arr = train_evaluate(
            X_sku, y, weeks, fcols, test_size)
        forecasts = forecast_future(results, X_sku, fcols, n_periods)

        # Pick the model with the lowest average error on the test weeks
        best_model = min(results, key=lambda k: results[k]["mae"])
        best = results[best_model]
        fc_best = forecasts[best_model]

        # Show SKU summary: category, color, avg price, avg sales, promo rate
        sm = sku_meta[sku_meta.sku == sku_id].iloc[0]
        display(HTML(f"<h3>SKU {sku_id} — {sm.functionality}</h3>"
                     f"<p style='color:#64748b;'>Color: {sm.color} · Avg price: ${sm.avg_price} · "
                     f"Avg sales: {sm.avg_sales}/wk · Promo rate: {sm.feat_rate}%</p>"))

        # Display key performance indicators: best model, error, R², and forecast vs recent trend
        avg_hist = np.mean(y[-12:])
        avg_fc = np.mean(fc_best["predictions"])
        delta = ((avg_fc - avg_hist) / max(avg_hist, 1)) * 100
        delta_sym = "▲" if delta >= 0 else "▼"
        display(metric_row([
            metric_card("Best model", best_model.split("(")[0].strip(), "Lowest MAE on test"),
            metric_card("Test MAE", f"{best['mae']:.1f}", "units/week"),
            metric_card("Test R²", f"{best['r2']:.3f}", "Good ✓" if best['r2'] > 0.5 else "Moderate"),
            metric_card(f"Avg forecast ({n_periods}w)", f"{avg_fc:.0f}", f"{delta_sym} {abs(delta):.1f}% vs recent"),
        ]))

        # Plot the timeline: past sales, test actuals, model predictions, and future forecast
        fig = plot_forecast(w_train, y_train, w_test_arr, y_test, results, forecasts, best_model, sku_id, n_periods)
        fig.show()

        # Build a week-by-week forecast table with predictions from each model and confidence range
        last_date = pd.Timestamp(w_test_arr[-1])
        rows = []
        for step in range(n_periods):
            wk = last_date + pd.Timedelta(weeks=step + 1)
            row = {"Week": wk.strftime("%Y-%m-%d")}
            for name in results:
                row[name] = f"{forecasts[name]['predictions'][step]:.0f}"
            row["95% CI (best)"] = f"[{fc_best['ci_lower'][step]:.0f}, {fc_best['ci_upper'][step]:.0f}]"
            rows.append(row)
        display(HTML("<h4>Forecast detail</h4>"))
        display(pd.DataFrame(rows))

        # Side-by-side bar chart: which model had the lowest error and best fit?
        display(HTML("<h4>Model comparison</h4>"))
        comp = plot_model_comparison(results)
        comp.show()

        # Detailed accuracy table: MAE, RMSE, MAPE%, R² for each model (★ = best)
        rows_m = []
        for name, res in results.items():
            rows_m.append({"Model": name, "MAE": f"{res['mae']:.2f}", "RMSE": f"{res['rmse']:.2f}",
                          "MAPE %": f"{res['mape']:.1f}", "R²": f"{res['r2']:.3f}",
                          "Best": "★" if name == best_model else ""})
        display(pd.DataFrame(rows_m))

        # Residual plot (where did the model over/under-predict?) and feature importance ranking
        display(HTML("<h4>Diagnostics</h4>"))
        plot_residuals(w_test_arr, y_test, results, best_model).show()
        plot_feature_importance(results, fcols).show()

_forecast_btn.on_click(_run_forecast)
display(widgets.VBox([
    widgets.HBox([_forecast_sku, _forecast_horizon, _forecast_test, _forecast_btn]),
    _forecast_output,
]))


## All-SKU forecast heatmap
Quick Random Forest forecast across all 44 SKUs.

In [8]:
# Forecast the next 4 weeks for every SKU using Random Forest (4 features, per-SKU)
n_periods = 4
all_preds = {}

for sku_id in sorted(proc['sku'].unique()):
    X_hm, y_hm, _, fcols_hm = build_demand_features(proc, sku_id)
    if len(X_hm) < 20:
        continue
    model = RandomForestRegressor(**TUNED_RF, random_state=42, n_jobs=-1)
    model.fit(X_hm, y_hm)
    last_row = X_hm[-1].copy().reshape(1, -1)
    preds = []
    for step in range(1, n_periods + 1):
        row = last_row.copy()
        row[0, fcols_hm.index('trend')] = X_hm[-1, fcols_hm.index('trend')] + step / len(X_hm)
        preds.append(max(model.predict(row)[0], 0))
    all_preds[sku_id] = preds

sku_labels = [f'SKU {s}' for s in all_preds.keys()]
z = np.array(list(all_preds.values()))
last_date = pd.Timestamp(proc['week'].max())
week_labels = [(last_date + pd.Timedelta(weeks=i)).strftime('%d %b') for i in range(1, n_periods + 1)]

fig = go.Figure(data=go.Heatmap(
    z=z, x=week_labels, y=sku_labels, colorscale='Blues',
    text=np.round(z, 0), texttemplate='%{text:.0f}', textfont=dict(size=9),
    hovertemplate='SKU: %{y}<br>Week: %{x}<br>Demand: %{z:.0f}<extra></extra>',
))
fig.update_layout(title='Forecasted demand — all SKUs (Random Forest, 4 features)',
                  height=max(500, len(sku_labels) * 22),
                  margin=dict(l=70, r=20, t=50, b=30), yaxis=dict(dtick=1))
fig.show()
print(f'Forecasted {len(all_preds)} SKUs × {n_periods} weeks')

Forecasted 44 SKUs × 4 weeks


## Methodology

**Data:** 44 SKUs × 98 weeks from the processed dataset with lagged prices, trend, month dummies, and one-hot categoricals.

**Train/test split:** Last N weeks held out (temporal split — no leakage).

**Feature selection (per-SKU):** With ~90 training rows and 20+ raw features, models risk overfitting — producing negative R² where even the mean would predict better. A two-stage selection runs for each SKU:
- **Stage 1 — Drop zero-variance columns:** One-hot categoricals that are constant within a SKU (e.g. `color_black` = all 1s for a black product) carry no information.
- **Stage 2 — RF importance ranking:** Core business variables (price, trend, promo flag, lagged prices) are always kept. Remaining features are ranked by a quick Random Forest's feature importance, and only those exceeding a 1% importance threshold are retained, up to K = min(10, n_train ÷ 10) total. This ensures near-zero-contribution features (e.g. month dummies at 0.3% importance) are discarded rather than adding noise.

**Hyperparameter tuning:** Random Forest and MLP hyperparameters were selected via `GridSearchCV` with `TimeSeriesSplit` (3 folds) on a representative subset of 8 SKUs spanning the sales volume distribution. This ensures temporal ordering is respected during validation (no future data leaks into training folds). The grid searched over:
- **Random Forest:** `n_estimators` ∈ {100, 200, 300}, `max_depth` ∈ {4, 6, 8, 12}, `min_samples_leaf` ∈ {2, 4, 8} — balancing ensemble size against overfitting risk on ~90 training rows per SKU.
- **MLP:** `hidden_layer_sizes` ∈ {(32,), (64,32), (128,64)}, `learning_rate_init` ∈ {0.001, 0.005, 0.01} — calibrating network capacity and convergence speed.
- **Linear Regression:** No tuning needed — OLS is the unique closed-form solution.

The winning hyperparameters (lowest cross-validated MAE) are then applied across all 44 SKUs.

**Models:**
- **Linear Regression** — OLS baseline on raw features
- **Random Forest** — Tuned via CV (see above)
- **Neural Network (MLP)** — Tuned via CV, StandardScaler-transformed inputs, early stopping with 15% validation hold-out

**Confidence intervals:** Training residual σ × 1.96 × √(step) — widens with forecast horizon. 95% level.

**Best model selection:** Lowest MAE on the held-out test set.

**References:**
- Hyndman, R.J. & Athanasopoulos, G. (2021). *Forecasting: Principles and Practice*, 3rd edition. OTexts.
- Cohen, M. C., Gras, P.E., Pentecoste, A., & Zhang, R. (2022). *Demand Prediction in Retail.* Springer.